# Experiments Master Notebook

**Paper:** *A Unified Maximum-Likelihood Framework for Signal Reconstruction:*  
*From Optimal Filtering to Noise-Aware Linear Autoencoders*

Consolidated notebook replacing all individual `block_0X` notebooks.

**Sections:**  
0. Setup  
1. Experiment-to-claim map  
2. Preprocessing contract & PSD audit  
3. E1 — Theorem 1: rank-1 EMPCA ≡ Optimal Filter  
4. E2 — Theorem 2: noise-aware AE ≡ EMPCA  
5. E3 — χ²(k) monotone / rank saturation  
6. E4 — Cramér-Rao bound  
7. E5 — Energy-resolution scaling (error bars)  
8. E6 — Noise-aware vs isotropic PCA ablation (error bars)  
9. E7 — Template mismatch bias curve  
10. E8 — Time-shift optimal filter  
11. E9 — EMPCA convergence  
12. Figure summary  

**Workflow:**  
Run `python implementation/generate_data.py` first (populates `results/data_cache.h5`,  
8 seeds × all experiments, ~20-40 min).  
If the cache is absent the notebook falls back to a single-seed inline run automatically.


## 0 — Setup

In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import h5py

warnings.filterwarnings('ignore', category=RuntimeWarning)

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / 'QP_simulator').exists() and (p / 'implementation').exists():
            return p
    raise RuntimeError(f'Cannot locate repo root from {start}')

REPO    = find_repo_root()
for _p in [REPO, REPO/'src', REPO/'QP_simulator', REPO/'implementation']:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

CACHE_H5 = REPO / 'results' / 'data_cache.h5'
FIG_DIR  = REPO / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root :', REPO)
print('Cache h5  :', CACHE_H5, '— exists:', CACHE_H5.exists())


In [ ]:
mpl.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 11, 'axes.labelsize': 11,
    'figure.dpi': 150, 'lines.linewidth': 1.6,
    'axes.grid': True, 'grid.alpha': 0.3,
})
BLUE, ORANGE, GREEN, RED = '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'

def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print(f'  saved -> {path.relative_to(REPO)}')

def load_df(h5, path):
    grp = h5[path]
    d = {}
    for k in grp.keys():
        v = grp[k][()]
        if v.dtype.kind == 'S':
            v = np.array([x.decode() for x in v])
        d[k] = v
    return pd.DataFrame(d)

def cache_groups(h5, prefix):
    return sorted(h5[prefix].keys()) if prefix in h5 else []

def mean_std(arr):
    mu = arr.mean(0)
    sd = arr.std(0, ddof=1) if arr.shape[0] > 1 else np.zeros_like(mu)
    return mu, sd

print('Helpers ready.')


In [ ]:
# Single-seed fallback when cache is absent
if not CACHE_H5.exists():
    print('Cache not found — running single-seed fallback (~2-5 min)...')
    from notebook_support import (
        CanonicalConfig, run_block04_theorem_suite,
        run_block05_bridge_suite, run_block06_convergence_suite,
        run_block07_ablation_suite,
    )
    _cfg = CanonicalConfig(seed=314159, sim_events_medium=200,
                           sim_events_large=200).validate()
    _fb = {
        'b04': run_block04_theorem_suite(_cfg),
        'b05': run_block05_bridge_suite(_cfg),
        'b06': run_block06_convergence_suite(_cfg),
        'b07': run_block07_ablation_suite(_cfg),
    }
    print('Fallback done.')
else:
    _fb = None
    print('Using h5 cache.')


---
## 1 — Experiment-to-claim map

Every plot in this notebook is anchored to a specific theorem, proposition,
or empirical claim in the paper.  The table below is the authoritative mapping
(generated from `notebook_support.dataframe_from_claim_map`).

### Design rules

- Each experiment carries a **support regime**: `theorem-support`, `real-support`, or `robustness-support`.
- Acceptance criteria are **explicit numeric thresholds** — no vague phrases.
- Structured-noise studies (E10, E11) are `robustness-support` unless there is a specific reason otherwise.

### Naming conventions

| Concept | Convention |
|---|---|
| Simulated rank-1 batch | `SIM-rank1` |
| Simulated controlled family | `SIM-controlled-family` |
| Rank label | `k{integer}` |
| Seed label | `seed_{integer}` |
| Figures | `results/figures/*.pdf` |
| Cache | `results/data_cache.h5` |


In [ ]:
from notebook_support import dataframe_from_claim_map, CanonicalConfig, config_as_frame

claim_map = dataframe_from_claim_map()
pd.set_option('display.max_colwidth', 90)
display(claim_map[['experiment_id', 'claim', 'paper_anchor', 'support_regime', 'metrics']])
print(f'\n{len(claim_map)} experiments mapped.')


In [ ]:
# Canonical config — single source of truth for all hyperparameters
cfg = CanonicalConfig().validate()
display(config_as_frame(cfg))


---
## 2 — Preprocessing contract and PSD audit

Freezes the analysis interface shared by all experiments.  
Requires real K-alpha data (`data/k_alpha_traces.h5`).  
If that file is absent, falls back to showing the analytic pink PSD used for simulations.

### Preprocessing decisions

| Decision | Value |
|---|---|
| Baseline window | pretrigger samples (default 4000) |
| Baseline method | per-event mean subtraction |
| Trace length | 32768 samples |
| Train / val / test split | 60 / 20 / 20, seed-fixed |
| Frequency transform | one-sided `rfft`, normalised by N |
| PSD | Welch on pretrigger windows; floored at 1st percentile |
| Weights | `w_k = 1 / J_k`; DC bin zeroed |
| Template normalisation | peak-aligned; unit peak |
| Whitening | element-wise division by `sqrt(J_k)` |
| Covariance model | diagonal (Toeplitz + stationarity) |


In [ ]:
try:
    from notebook_support import run_block02_audit
    _audit = run_block02_audit(cfg)
    bundle = _audit['bundle']
    display(_audit['audit_df'])
    display(_audit['reports_df'])

    fig, ax = plt.subplots(figsize=(6, 3.8))
    freqs = bundle.metadata['empirical_psd_freqs']
    emp   = bundle.metadata['empirical_psd']
    ax.loglog(freqs[1:], bundle.psd_one_sided[1:], color=BLUE,
              label='canonical PSD')
    ax.loglog(freqs[1:], emp[1:], color=ORANGE, ls='--',
              label='empirical pretrigger PSD')
    ax.set_xlabel('frequency [Hz]'); ax.set_ylabel('PSD')
    ax.set_title('PSD audit — real K-alpha data')
    ax.legend(fontsize=9); fig.tight_layout()
    save_fig(fig, 'psd_audit.pdf'); plt.show()

except FileNotFoundError as _e:
    print(f'Real data not found ({_e}).')
    print('Showing analytic pink PSD used for simulations instead.')
    from notebook_support import stationary_noise_generator
    ng = stationary_noise_generator(cfg, noise_type='pink', noise_power=1.0)
    freqs, psd = ng.build_psd(cfg.trace_len)
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.loglog(freqs[1:], psd[1:], color=BLUE, label='analytic pink PSD (simulation)')
    ax.set_xlabel('frequency [Hz]'); ax.set_ylabel('PSD')
    ax.set_title('Simulation PSD (real K-alpha data absent)')
    ax.legend(fontsize=9); fig.tight_layout(); plt.show()


---
## 3 — E1: Theorem 1 — rank-1 EMPCA ≡ Optimal Filter

*Paper §4.3, Table 1 in §3.*

**Pass criteria:** ρ_w > 0.9999 · amp correlation > 0.999 · median rel error < 1e-3 · KS p > 0.05


In [ ]:
e1_rows = []
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e1_e4_e5'):
            path = f'e1_e4_e5/{sg}/e1'
            if path in h5:
                df = load_df(h5, path); df['seed'] = sg; e1_rows.append(df)

if not e1_rows and _fb:
    e1_rows = [_fb['b04']['rank1_summary_df'].assign(seed='fallback')]

e1_all = pd.concat(e1_rows, ignore_index=True)

THRESHOLDS = {
    'weighted_subspace_cosine': (0.9999, '>='),
    'amplitude_correlation':    (0.999,  '>='),
    'median_relative_error':    (1e-3,   '<='),
    'residual_ks_pvalue':       (0.05,   '>='),
}
numeric = [c for c in THRESHOLDS if c in e1_all.columns]
smry = e1_all[numeric].agg(['mean', 'std']).T
for col, (thr, op) in THRESHOLDS.items():
    if col not in smry.index: continue
    mu = smry.loc[col, 'mean']
    smry.loc[col, 'threshold'] = thr
    smry.loc[col, 'pass'] = 'PASS' if (mu >= thr if op == '>=' else mu <= thr) else 'FAIL'
display(smry)


---
## 4 — E2: Theorem 2 — noise-aware linear AE ≡ EMPCA

*Paper §5.3.*  
**Pass criterion:** minimum principal-angle cosine > 0.9999 for k = 1, 2, 3.


In [ ]:
e2_by_k = {}
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e2'):
            path = f'e2/{sg}'
            if path not in h5: continue
            df = load_df(h5, path)
            if 'k' not in df.columns: continue
            for _, row in df.iterrows():
                e2_by_k.setdefault(int(row['k']), []).append(
                    float(row['min_principal_cosine']))

if not e2_by_k and _fb:
    bd = _fb['b05']['bridge_df'].copy()
    bd['min_pc'] = bd['principal_angle_cosines'].map(
        lambda x: min(x) if hasattr(x, '__iter__') else x)
    for _, r in bd.iterrows():
        e2_by_k.setdefault(int(r['k']), []).append(float(r['min_pc']))

ks    = sorted(e2_by_k)
mu_e2 = np.array([np.mean(e2_by_k[k]) for k in ks])
sd_e2 = np.array([np.std(e2_by_k[k], ddof=1) if len(e2_by_k[k]) > 1
                  else 0. for k in ks])

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.errorbar(ks, mu_e2, yerr=sd_e2, fmt='o-', color=BLUE, capsize=4,
            label='min cosine (mean +/- std)')
ax.axhline(0.9999, color='gray', ls='--', lw=1, label='pass threshold 0.9999')
ax.set_xlabel('rank k'); ax.set_ylabel('minimum principal-angle cosine')
ax.set_ylim(0.998, 1.0005)
ax.set_title('E2 -- EMPCA vs exact weighted bridge')
ax.legend(fontsize=9); fig.tight_layout()
save_fig(fig, 'e2_bridge_cosines.pdf'); plt.show()
print('All k pass:', all(m >= 0.9999 for m in mu_e2))


---
## 5 — E3: Rank saturation — χ²(k) monotone decrease

*Paper §4.4.*  
χ² must strictly decrease from k=1 to k=2 on the multi-dimensional family, then plateau.


In [ ]:
e3_by_seed = {}
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e3_e9'):
            path = f'e3_e9/{sg}/rank_summary'
            if path in h5:
                e3_by_seed[sg] = load_df(h5, path)

if not e3_by_seed and _fb:
    e3_by_seed['fallback'] = (
        _fb['b06']['rank_summary_df'][['k', 'chi2_proxy_mean', 'chi2_proxy_std']]
        .copy())

ks_e3   = sorted(list(e3_by_seed.values())[0]['k'].astype(int))
chi2stk = np.stack([df.sort_values('k')['chi2_proxy_mean'].values
                    for df in e3_by_seed.values()], axis=0)
mu_c, sd_c = mean_std(chi2stk)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.errorbar(ks_e3, mu_c, yerr=sd_c, fmt='o-', color=BLUE, capsize=4,
            label='held-out chi2 (mean +/- std)')
ax.set_xlabel('rank k'); ax.set_ylabel('held-out weighted residual')
ax.set_title('E3 -- Rank saturation (multi-dimensional family)')
ax.legend(fontsize=9); fig.tight_layout()
save_fig(fig, 'e3_rank_saturation.pdf'); plt.show()
if len(mu_c) > 1:
    print(f'Delta chi2(1->2) / chi2(1) = {(mu_c[0]-mu_c[1])/mu_c[0]:.4f}')


---
## 6 — E4: Cramér-Rao bound verification

*Paper §3.2.*  
**Pass criterion:** |σ²_emp − 1/N_Φ| / (1/N_Φ) < 5% for each noise colour.


In [ ]:
e4_rows = []
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e1_e4_e5'):
            path = f'e1_e4_e5/{sg}/e4'
            if path in h5:
                df = load_df(h5, path); df['seed'] = sg; e4_rows.append(df)

if not e4_rows and _fb:
    e4_rows = [_fb['b04']['crb_df'].assign(seed='fallback')]

e4_all = pd.concat(e4_rows, ignore_index=True)
e4_agg = (e4_all.groupby('noise_type')['relative_error']
          .agg(['mean', 'std']).reset_index()
          .rename(columns={'mean': 'mean_rel_error', 'std': 'std_rel_error'}))
e4_agg['pass'] = e4_agg['mean_rel_error'].apply(
    lambda x: 'PASS' if x < 0.05 else 'FAIL')
display(e4_agg)

fig, ax = plt.subplots(figsize=(5, 3.5))
x = np.arange(len(e4_agg))
ax.bar(x, e4_agg['mean_rel_error'], yerr=e4_agg['std_rel_error'].fillna(0),
       capsize=5, color=BLUE, alpha=0.8,
       error_kw={'ecolor': 'black', 'lw': 1.5})
ax.axhline(0.05, color='red', ls='--', lw=1, label='5% threshold')
ax.set_xticks(x); ax.set_xticklabels(e4_agg['noise_type'])
ax.set_ylabel('|sigma2_emp - 1/N_Phi| / (1/N_Phi)')
ax.set_title('E4 -- CRB verification'); ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'e4_crb_verification.pdf'); plt.show()


---
## 7 — E5: Energy-resolution scaling law

*Paper §3.2.*  
σ_E ∝ (noise_power)^{1/2}.  Log-log slope should be 0.500 ± 0.05.  
Error bars = std across 8 seeds.


In [ ]:
e5_emp = {}; e5_pred = {}; e5_pwr = {}
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e1_e4_e5'):
            path = f'e1_e4_e5/{sg}/e5'
            if path not in h5: continue
            df = load_df(h5, path)
            e5_pwr[sg]  = df['noise_power'].values.astype(float)
            e5_emp[sg]  = df['sigma_emp'].values.astype(float)
            e5_pred[sg] = df['sigma_pred'].values.astype(float)

if not e5_emp and _fb:
    df = _fb['b04']['resolution_df']
    e5_pwr['fallback']  = df['noise_power'].values.astype(float)
    e5_emp['fallback']  = df['sigma_emp'].values.astype(float)
    e5_pred['fallback'] = df['sigma_pred'].values.astype(float)

powers         = list(e5_pwr.values())[0]
mu_emp, sd_emp = mean_std(np.stack(list(e5_emp.values()),  axis=0))
mu_prd, sd_prd = mean_std(np.stack(list(e5_pred.values()), axis=0))
slopes = [np.polyfit(np.log(p), np.log(s), 1)[0]
          for p, s in zip(e5_pwr.values(), e5_emp.values())]
slope_mu = float(np.mean(slopes))
slope_sd = float(np.std(slopes))

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.errorbar(powers, mu_emp, yerr=sd_emp, fmt='o-', color=BLUE,
            capsize=4, label='sigma_emp (mean +/- std)')
ax.errorbar(powers, mu_prd, yerr=sd_prd, fmt='s--', color=ORANGE,
            capsize=4, label='sigma_pred = 1/sqrt(N_Phi)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('noise power'); ax.set_ylabel('sigma_E proxy')
ax.set_title('E5 -- Energy-resolution scaling')
ax.legend(fontsize=9)
ax.text(0.05, 0.95,
        f'log-log slope = {slope_mu:.3f} +/- {slope_sd:.3f}  (ideal 0.500)',
        transform=ax.transAxes, va='top', fontsize=9)
fig.tight_layout()
save_fig(fig, 'e5_resolution_scaling.pdf'); plt.show()
print(f'slope = {slope_mu:.4f}  pass: {abs(slope_mu - 0.5) < 0.05}')


---
## 8 — E6: Noise-aware EMPCA vs isotropic PCA

*Paper §7 — main practical result.*  
Expected: white noise → ~0 gap; pink/brownian → significant improvement.  
Error bars = std across seeds.


In [ ]:
e6_by_seed = {}
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e6_e7'):
            path = f'e6_e7/{sg}/e6'
            if path in h5:
                e6_by_seed[sg] = load_df(h5, path)

if not e6_by_seed and _fb:
    e6_by_seed['fallback'] = _fb['b07']['ablation_df'].copy()

noise_types = list(e6_by_seed.values())[0]['noise_type'].tolist()
ri_stk = np.stack(
    [df.set_index('noise_type').loc[noise_types, 'relative_improvement'].values
     for df in e6_by_seed.values()], axis=0)
mu_ri, sd_ri = mean_std(ri_stk)

fig, ax = plt.subplots(figsize=(5.5, 4))
x = np.arange(len(noise_types))
bar_colors = [BLUE, ORANGE, GREEN][:len(noise_types)]
ax.bar(x, mu_ri, yerr=sd_ri, capsize=6, color=bar_colors, alpha=0.85,
       error_kw={'ecolor': 'black', 'lw': 1.5})
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(noise_types)
ax.set_ylabel('relative improvement  (chi2_iso - chi2_w) / chi2_iso')
ax.set_title('E6 -- Noise-aware vs isotropic loss')
for xi, (m, s) in enumerate(zip(mu_ri, sd_ri)):
    ax.text(xi, m + s + 0.005, f'{m:.3f}', ha='center', fontsize=8)
fig.tight_layout()
save_fig(fig, 'e6_ablation.pdf'); plt.show()
display(pd.DataFrame({'noise_type': noise_types,
                      'mean_improvement': mu_ri, 'std': sd_ri}))


---
## 9 — E7: Template mismatch bias

*Paper §3.4.*  
When τ_decay differs from the nominal 3 ms, fixed-template OF is biased.  
Rank-2 EMPCA recovers amplitude.

**Fix vs original block_07:** x-axis is τ (ms), not cos²(θ_w).  
Dual y-axes show the OF estimate and the subspace overlap as independent functions of τ.


In [ ]:
e7_curve = {}
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e6_e7'):
            path = f'e6_e7/{sg}/e7/curve'
            if path in h5:
                e7_curve[sg] = load_df(h5, path)

if not e7_curve and _fb:
    e7_curve['fallback'] = _fb['b07']['mismatch_curve_df'].copy()

ref    = list(e7_curve.values())[0].sort_values('tau_decay')
tau_ms = ref['tau_decay'].values / 1e6
cos2   = ref['cosine_squared'].values
of_est = ref['of_estimate_for_unit_shape'].values

fig, ax1 = plt.subplots(figsize=(6, 4))
ax2 = ax1.twinx()
ax1.plot(tau_ms, of_est, 'o-',  color=BLUE,   label='OF estimate (unit-shape input)')
ax2.plot(tau_ms, cos2,   's--', color=ORANGE, label='cos2(theta_w) subspace overlap')
ax1.axvline(3.0, color='gray', ls=':', lw=1)
ylo, yhi = ax1.get_ylim()
ax1.text(3.05, ylo + 0.02*(yhi-ylo), 'nominal tau = 3 ms',
         fontsize=8, color='gray')
ax1.set_xlabel('tau_decay (ms)')
ax1.set_ylabel('OF amplitude estimate (unit-shape input)', color=BLUE)
ax2.set_ylabel('cos2(theta_w)', color=ORANGE)
ax1.tick_params(axis='y', colors=BLUE)
ax2.tick_params(axis='y', colors=ORANGE)
l1, lb1 = ax1.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lb1+lb2, fontsize=8, loc='lower right')
ax1.set_title('E7 -- Template mismatch bias curve')
fig.tight_layout()
save_fig(fig, 'e7_mismatch_curve.pdf'); plt.show()


In [ ]:
# E7 method summary: OF vs EMPCA k=1 vs k=2
e7_rows = []
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e6_e7'):
            path = f'e6_e7/{sg}/e7'
            if path in h5:
                df = load_df(h5, path); df['seed'] = sg; e7_rows.append(df)

if not e7_rows and _fb:
    e7_rows = [_fb['b07']['mismatch_df'].assign(seed='fallback')]

if e7_rows:
    e7_all = pd.concat(e7_rows, ignore_index=True)
    display(e7_all.groupby('method')[['mean_relative_bias', 'rmse']]
            .agg(['mean', 'std']))


---
## 10 — E8: Time-shift optimal filter

*Paper §3.2.*  
Time-shift OF scans a grid of shifted templates.  Arrival-time RMSE and amplitude bias should both improve over fixed-template OF.


In [ ]:
e8_rows = []
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e6_e7'):
            path = f'e6_e7/{sg}/e8'
            if path in h5:
                df = load_df(h5, path); df['seed'] = sg; e8_rows.append(df)

if not e8_rows and _fb:
    e8_rows = [_fb['b07']['shift_df'].assign(seed='fallback')]

if e8_rows:
    e8_all = pd.concat(e8_rows, ignore_index=True)
    display(e8_all.groupby('method')[['amplitude_rmse', 'mean_relative_bias']]
            .agg(['mean', 'std']))


---
## 11 — E9: EMPCA convergence

*Paper §6.1.*  
χ² must decrease monotonically; convergence expected within O(10-50) iterations.  
Shaded band = std across seeds.


In [ ]:
e9_dfs = []
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e3_e9'):
            path = f'e3_e9/{sg}/e9'
            if path in h5:
                df = load_df(h5, path); df['seed'] = sg; e9_dfs.append(df)

if not e9_dfs and _fb:
    e9_dfs = [_fb['b06']['convergence_df'].assign(seed='fallback')]

e9_all = pd.concat(e9_dfs, ignore_index=True)
e9_all['chi2_norm'] = e9_all.groupby(['seed','k','init'])['chi2'].transform(
    lambda s: s / s.iloc[0])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
kcols = {1: BLUE, 2: ORANGE, 3: GREEN}
for ax, init in zip(axes, ['random', 'svd']):
    sub = e9_all[(e9_all['init'] == init) & e9_all['k'].isin([1,2,3])]
    for k, grp in sub.groupby('k'):
        it = grp.groupby('iteration')['chi2_norm']
        mu = it.mean(); sd = it.std().fillna(0)
        ax.plot(mu.index.values, mu.values, color=kcols[k], label=f'k={k}')
        ax.fill_between(mu.index.values, mu-sd, mu+sd,
                        color=kcols[k], alpha=0.15)
    ax.set_xlabel('iteration'); ax.set_ylabel('normalised chi2')
    ax.set_title(f'init = {init}'); ax.legend(fontsize=9)
fig.suptitle('E9 -- EMPCA convergence  (shading = std across seeds)')
fig.tight_layout()
save_fig(fig, 'e9_convergence.pdf'); plt.show()


In [ ]:
e9_smry_dfs = []
if CACHE_H5.exists():
    with h5py.File(CACHE_H5, 'r') as h5:
        for sg in cache_groups(h5, 'e3_e9'):
            path = f'e3_e9/{sg}/e9/summary'
            if path in h5:
                df = load_df(h5, path); df['seed'] = sg; e9_smry_dfs.append(df)

if e9_smry_dfs:
    e9_smry = pd.concat(e9_smry_dfs, ignore_index=True)
    display(e9_smry.groupby(['k','init'])
            [['iterations_used','iter_to_relative_tol_1e-6']]
            .agg(lambda x: f'{x.mean():.1f} +/- {x.std():.1f}'))
elif _fb:
    display(_fb['b06']['convergence_summary_df'])


---
## 12 — Figure summary

All figures saved to `results/figures/`.

| File | Experiment | Paper section |
|---|---|---|
| `psd_audit.pdf` | Real-data PSD audit | §2 preprocessing |
| `e2_bridge_cosines.pdf` | E2: Theorem 2 (Bridge) | §5.3 |
| `e3_rank_saturation.pdf` | E3: χ²(k) monotone | §4.4 |
| `e4_crb_verification.pdf` | E4: CRB | §3.2 |
| `e5_resolution_scaling.pdf` | E5: σ_E scaling (error bars) | §3.2 |
| `e6_ablation.pdf` | E6: noise-aware vs iso (error bars) | §7 |
| `e7_mismatch_curve.pdf` | E7: template mismatch (fixed x-axis) | §3.4 |
| `e9_convergence.pdf` | E9: convergence (shaded band) | §6.1 |


In [ ]:
print('Figures in:', FIG_DIR)
for f in sorted(FIG_DIR.glob('*.pdf')):
    print(' ', f.name)
